# Patch-Based Inference

> Sliding-window inference for 3D medical image segmentation using fastMONAI's `PatchInferenceEngine`.

This tutorial demonstrates how to perform inference on trained patch-based models:

1. **GridSampler**: Extracts overlapping patches from the input volume
2. **Model**: Predicts on each patch
3. **GridAggregator**: Reconstructs the full volume with Hann windowing for smooth boundaries

**Prerequisites**: Run [12a_tutorial_patch_training.ipynb](12a_tutorial_patch_training.ipynb) first to train a model and save the configuration.

[![Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MMIV-ML/fastMONAI/blob/main/nbs/12b_tutorial_patch_inference.ipynb)

In [ ]:
#| hide
#Install `fastMONAI` if notebook is running on Google Colab
try:
    import google.colab
    %pip install fastMONAI
    from fastMONAI.utils import print_colab_gpu_info
    print_colab_gpu_info()
except:
    print('Running locally')

In [ ]:
from fastMONAI.vision_all import *

from monai.apps import DecathlonDataset
from sklearn.model_selection import train_test_split

### Load data

Load the same dataset and recreate the train/test split from the training notebook.

In [ ]:
path = Path('../data')
path.mkdir(exist_ok=True)

In [ ]:
task = "Task02_Heart"
training_data = DecathlonDataset(root_dir=path, task=task, section="training", 
    download=True, cache_num=0, num_workers=3)

In [ ]:
df = pd.DataFrame(training_data.data)
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)
print(f"Test samples: {len(test_df)}")
test_df

### Load configuration

Load the patch configuration saved during training. This ensures preprocessing consistency:

- **apply_reorder**: Whether to reorder to RAS+ orientation
- **target_spacing**: Target voxel spacing for resampling
- **patch_size, patch_overlap, aggregation_mode**: Inference parameters

> **Critical**: Mismatched preprocessing parameters will produce incorrect predictions (e.g., mirrored or rotated outputs).

In [ ]:
config_dict = load_patch_variables('patch_config.pkl')
print("Loaded configuration:")
for k, v in config_dict.items():
    print(f"  {k}: {v}")

In [ ]:
patch_config = PatchConfig(**config_dict)
apply_reorder = config_dict['apply_reorder']
target_spacing = config_dict['target_spacing']

### Recreate model and load weights

The model architecture must match exactly what was used during training.

In [ ]:
from monai.networks.nets import UNet
from monai.networks.layers import Norm
from monai.losses import DiceCELoss

In [ ]:
model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=2,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
    norm=Norm.INSTANCE
)

loss_func = CustomLoss(loss_func=DiceCELoss(
    to_onehot_y=True,
    softmax=True,
    include_background=True
))

Create a minimal DataLoader just to initialize the Learner and load weights.

In [ ]:
# Create minimal DataLoader for Learner (just to load weights)
mini_dls = MedPatchDataLoaders.from_df(
    df=train_df.head(2),
    img_col='image',
    mask_col='label',
    valid_pct=0.5,
    patch_config=patch_config,
    apply_reorder=apply_reorder,
    target_spacing=target_spacing,
    pre_patch_tfms=[ZNormalization()],
    bs=1
)

In [ ]:
learn = Learner(mini_dls, model, loss_func=loss_func)
learn.load('heart-patch-weights');

### Define pre-inference transforms

These transforms **MUST match** the `pre_patch_tfms` used during training.

> **Troubleshooting**: If predictions appear mirrored, rotated, or completely wrong, check that `apply_reorder`, `target_spacing`, and `pre_inference_tfms` match training.

In [ ]:
# MUST match training pre_patch_tfms
pre_inference_tfms = [ZNormalization()]

### Create PatchInferenceEngine

The engine handles the complete sliding-window inference pipeline:
1. Load and preprocess the image (reorder, resample, normalize)
2. Pad if image is smaller than patch size
3. Extract overlapping patches with GridSampler
4. Predict on batches of patches
5. Reconstruct full volume with GridAggregator using Hann windowing

In [ ]:
engine = PatchInferenceEngine(
    learner=learn,
    config=patch_config,
    apply_reorder=apply_reorder,
    target_spacing=target_spacing,
    pre_inference_tfms=pre_inference_tfms,
    batch_size=4
)

### Single image inference

Use `engine.predict()` to predict on a single image.

In [ ]:
test_path = test_df.iloc[0]['image']
pred, affine = engine.predict(test_path, return_affine=True)
print(f"Input: {test_path}")
print(f"Prediction shape: {pred.shape}")
print(f"Unique values: {torch.unique(pred).tolist()}")

### Batch inference

Use `patch_inference()` to predict on multiple images with optional NIfTI output.

In [ ]:
test_paths = test_df['image'].tolist()

predictions = patch_inference(
    learner=learn,
    config=patch_config,
    file_paths=test_paths,
    pre_inference_tfms=pre_inference_tfms,
    save_dir='predictions/patch_heart',
    progress=True
)

print(f"\nGenerated {len(predictions)} predictions")
print(f"Saved to: predictions/patch_heart/")

### Evaluate predictions

Compare predictions with ground truth to compute Dice scores.

In [ ]:
dice_scores = []

for i, pred in enumerate(predictions):
    gt_path = test_df.iloc[i]['label']
    
    # Load ground truth with same preprocessing
    gt = MedMask.create(gt_path, apply_reorder=apply_reorder, target_spacing=target_spacing)
    
    # Compute Dice score
    # pred shape: [1, D, H, W], gt.data shape: [1, D, H, W]
    dice = binary_dice_score(pred.unsqueeze(0).float(), gt.data.unsqueeze(0).float())
    dice_scores.append(float(dice))
    
    print(f"Sample {i+1}: {Path(test_df.iloc[i]['image']).name} - Dice = {dice:.4f}")

print(f"\nMean Dice: {np.mean(dice_scores):.4f}")

### Visualize predictions

In [ ]:
# Load and visualize first test case
test_img = MedImage.create(test_df.iloc[0]['image'], apply_reorder=apply_reorder, target_spacing=target_spacing)
test_gt = MedMask.create(test_df.iloc[0]['label'], apply_reorder=apply_reorder, target_spacing=target_spacing)
test_pred = MedMask(predictions[0])

# Plot middle slice
mid_slice = test_img.shape[-1] // 2

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(test_img.data[0, :, :, mid_slice].cpu().numpy(), cmap='gray')
axes[0].set_title('Input Image')
axes[0].axis('off')

axes[1].imshow(test_gt.data[0, :, :, mid_slice].cpu().numpy(), cmap='gray')
axes[1].set_title('Ground Truth')
axes[1].axis('off')

axes[2].imshow(test_pred.data[0, :, :, mid_slice].cpu().numpy(), cmap='gray')
axes[2].set_title(f'Prediction (Dice: {dice_scores[0]:.4f})')
axes[2].axis('off')

plt.tight_layout()
plt.show()

### Summary

In this tutorial, we demonstrated patch-based inference for 3D medical image segmentation:

1. **`load_patch_variables()`**: Load configuration saved during training
2. **`PatchInferenceEngine`**: Sliding-window inference with GridSampler and GridAggregator
3. **`patch_inference()`**: Batch inference with optional NIfTI output

**Key points**:
- Preprocessing parameters (`apply_reorder`, `target_spacing`, `pre_inference_tfms`) **MUST match** training
- Hann windowing (`aggregation_mode='hann'`) produces smooth boundaries between patches
- `keep_largest_component=True` can clean up small spurious predictions

**When to use patch-based inference**:
- Large images that don't fit in GPU memory
- Variable-sized inputs (no resizing needed)
- Memory-constrained environments

**Tradeoffs**:
- Slower than full-image inference (multiple forward passes)
- Overlap and aggregation add computational overhead
- But enables processing of arbitrarily large volumes